# Etapa 3 — SVM RBF em CPU e CUDA

**Inteligência Artificial · Ciência da Computação · PUC Minas · Trabalho Prático 3**

No Colab, escolha **Ambiente de execução → Alterar tipo de ambiente de execução → GPU** e, em seguida, **Executar tudo**. Forneça apenas o **treino.csv oficial** se for solicitado. Com uma GPU NVIDIA em Linux, o notebook prepara o cuML e executa na GPU as quatro divisões de comparação. Também roda uma divisão em CPU para medir o custo dos dois backends nas mesmas condições. Em uma sessão sem GPU compatível, roda a verificação de tolerância em CPU.

As duas versões usam o RBF escolhido no experimento anterior, com C=10 e sem pesos de classe. Usamos as mesmas matrizes em precisão dupla e o mesmo gamma calculado a partir de cada treino. O teste oficial continua reservado para a entrega final.

## O que a SVM ajusta

A SVM de margem suave minimiza $ \frac12\lVert w\rVert^2 + C\sum_i \max(0, 1-y_i f(x_i)) $. A penalidade é zero quando o exemplo fica do lado correto e fora da margem. Ela aumenta quando um exemplo se aproxima do lado errado da fronteira ou cruza a fronteira.

No RBF, o solver ajusta os coeficientes duais $\alpha_i$ dos exemplos de treino. O método SMO escolhe coeficientes em que ainda há violação das condições de otimalidade e faz ajustes que respeitam $0\leq\alpha_i\leq C$ e $\sum_i y_i\alpha_i=0$. Os exemplos com coeficientes diferentes de zero são os vetores de suporte. Já os pesos de classe, se forem usados, alteram o valor de C por classe.

As iterações do solver servem para resolver essa otimização. Com max_iter=-1, comparamos a tolerância de parada de 0,001 com a tolerância mais estrita de 0,0001. O cuML também tem um critério de parada por estagnação. O número de iterações de um solver tem significado próprio e serve para comparar tolerâncias dentro do mesmo backend. A métrica de classificação continua sendo calculada na validação.

[Formulação do SVC](https://scikit-learn.org/1.7/modules/svm.html#mathematical-formulation) ·
[Parâmetros do SVC CUDA](https://docs.nvidia.com/cuml/latest/api/generated/cuml.svm.SVC/)

In [ ]:
from pathlib import Path
import csv
import json
import os
import subprocess
import sys
import tempfile
from hashlib import sha256
from IPython.display import Markdown, display

candidatos = [
    Path("data/raw/treino.csv"),
    Path("treino.csv"),
    Path("/content/treino.csv"),
]
treino_csv = next((arquivo for arquivo in candidatos if arquivo.is_file()), None)
if treino_csv is None:
    try:
        from google.colab import files
    except ImportError as erro:
        raise FileNotFoundError(
            "Coloque o treino.csv oficial na pasta inicial ou em data/raw e execute novamente."
        ) from erro
    treino_csv = Path("data/raw/treino.csv")
    treino_csv.parent.mkdir(parents=True, exist_ok=True)
    print("Selecione o treino.csv oficial do Canvas.")
    files.upload_file(str(treino_csv))

treino_csv = treino_csv.resolve()
with treino_csv.open(encoding="utf-8-sig", newline="") as arquivo:
    cabecalho = next(csv.reader(arquivo), [])
obrigatorias = {"class", "protocol_type", "service", "flag", "src_bytes", "total_bytes"}
if not obrigatorias.issubset(cabecalho):
    raise ValueError(
        f"O arquivo em {treino_csv} não tem o cabeçalho do treino oficial. "
        "Substitua-o pelo treino.csv correto e execute novamente."
    )
trabalho = Path(tempfile.mkdtemp(prefix="ia-etapa3-svm-cuda-"))
codigo_dados = trabalho / "dados_etapa2.py"
codigo_tecnicas = trabalho / "tecnicas_etapa2.py"
codigo_experimentos = trabalho / "comparar_cpu_cuda.py"
print("Base encontrada:", treino_csv.name)

In [ ]:
VERSOES = {
    "numpy": "2.5.4",
    "pandas": "3.0.1",
    "scipy": "1.18.1",
    "scikit-learn": "1.7.2",
    "imbalanced-learn": "0.14.1",
    "joblib": "1.6.0",
    "threadpoolctl": "3.8.0",
    "sklearn-compat": "0.1.6",
}

def consultar_versoes(ambiente):
    consulta = (
        "import importlib.metadata as m, json, sys; "
        "print(json.dumps({n: next((d.version for d in m.distributions(name=n)), None) "
        "for n in sys.argv[1:]}))"
    )
    resposta = subprocess.check_output(
        [sys.executable, "-c", consulta, *VERSOES], env=ambiente, text=True
    )
    return json.loads(resposta)

ambiente = os.environ.copy()
ambiente["PYTHONIOENCODING"] = "utf-8"
ambiente["PYTHONNOUSERSITE"] = "1"
ambiente["OPENBLAS_NUM_THREADS"] = "1"
ambiente["OMP_NUM_THREADS"] = "1"
if consultar_versoes(ambiente) != VERSOES:
    assinatura = sha256(json.dumps(VERSOES, sort_keys=True).encode()).hexdigest()[:12]
    bibliotecas = Path(tempfile.gettempdir()) / f"ia-etapa2-{sys.implementation.cache_tag}-{assinatura}"
    caminho_existente = ambiente.get("PYTHONPATH")
    ambiente["PYTHONPATH"] = os.pathsep.join(filter(None, (str(bibliotecas), caminho_existente)))
    if consultar_versoes(ambiente) != VERSOES:
        print("Instalando as versões dos experimentos. Aguarde a conclusão desta célula.", flush=True)
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "--disable-pip-version-check",
             "--only-binary=:all:", "--upgrade", "--target", str(bibliotecas),
             *[f"{nome}=={versao}" for nome, versao in VERSOES.items()]],
            check=True, env={**os.environ, "PYTHONNOUSERSITE": "1"},
        )
versoes_ativas = consultar_versoes(ambiente)
if versoes_ativas != VERSOES:
    raise RuntimeError(f"As versões do ambiente não coincidiram com as previstas: {versoes_ativas}")
print("Python:", sys.version.split()[0])
print("Bibliotecas:", versoes_ativas)

## Divisões e pré-processamento

Usamos as divisões agrupadas da Etapa 2. O preparo ajusta mediana, one-hot e Z-score numérico no treino de cada divisão, após retirar as duplicatas de mesmo rótulo. CPU e GPU recebem os mesmos arrays. A quinta divisão fica fora desta comparação.

In [ ]:
%%writefile "{codigo_dados}"
"""Prepara os dados e cria divisoes agrupadas para as etapas 2 e 3."""

import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import StratifiedGroupKFold

ALVO = "class"
COLUNAS_EXCLUIDAS_EXPERIMENTOS = (
    "num_outbound_cmds", "tempo_resposta_analista_min", "tipo_alerta_soc", "score_ids_legado",
)
SEMENTE = 42
NOMINAIS = ["protocol_type", "service", "flag", "protocolo_cod"]
COPIAS = ["src_bytes_kb", "dst_bytes_kb", "same_srv_pct", "protocolo_cod"]


def carregar_treino(caminho: str | Path) -> pd.DataFrame:
    """Lê o CSV oficial e valida a coluna alvo."""
    caminho = Path(caminho)
    if not caminho.is_file():
        raise FileNotFoundError(f"Base oficial não encontrada: {caminho}")
    dados = pd.read_csv(caminho)
    if ALVO not in dados or dados.empty:
        raise ValueError(f"A base deve ser não vazia e conter {ALVO!r}.")
    if dados[ALVO].isna().any() or not set(dados[ALVO].unique()).issubset({"normal", "anomaly"}):
        raise ValueError("A coluna class deve conter somente normal ou anomaly, sem ausências.")
    return dados.copy()


def identificar_taxas(dados: pd.DataFrame) -> list[str]:
    """Retorna apenas frações ``*_rate``; ``*_pct`` usa escala percentual."""
    return [c for c in dados.select_dtypes(include="number") if c.lower().endswith("rate")]


def preparar_base(dados):
    entradas = dados.drop(columns=["class", *COLUNAS_EXCLUIDAS_EXPERIMENTOS])
    for coluna in NOMINAIS:
        entradas[coluna] = entradas[coluna].astype("string").str.strip().str.casefold().replace("", pd.NA)

    invalidos = {}
    for coluna in ("duration", "src_bytes", "dst_bytes"):
        invalidos[coluna] = entradas[coluna].lt(0)
    for coluna in identificar_taxas(entradas):
        invalidos[coluna] = entradas[coluna].notna() & ~entradas[coluna].between(0, 1)
    invalidos["src_bytes_kb"] = invalidos["src_bytes"] | entradas["src_bytes_kb"].lt(0)
    invalidos["dst_bytes_kb"] = invalidos["dst_bytes"] | entradas["dst_bytes_kb"].lt(0)
    invalidos["total_bytes"] = invalidos["src_bytes"] | invalidos["dst_bytes"] | entradas["total_bytes"].lt(0)
    percentual = entradas["same_srv_pct"]
    invalidos["same_srv_pct"] = invalidos["same_srv_rate"] | (percentual.notna() & ~percentual.between(0, 100))
    for coluna, mascara in invalidos.items():
        entradas[coluna] = entradas[coluna].mask(mascara)

    representacoes = {
        "A": entradas,
        "B": entradas.drop(columns=COPIAS),
        "C": entradas.drop(columns=[*COPIAS, "total_bytes"]),
    }
    assert [m.shape[1] for m in representacoes.values()] == [45, 41, 40]
    grupos = representacoes["C"].groupby(list(representacoes["C"]), dropna=False, sort=False).ngroup()
    assert grupos.notna().all()
    return representacoes, grupos


def escolher_linhas(x, y, indices, regra):
    if regra == "todas":
        return indices
    chaves = x.iloc[indices].assign(rotulo=y[indices])
    mascara = np.ones(len(indices), dtype=bool)
    if regra in {"sem_conflitos", "sem_ambos"}:
        grupos = chaves.groupby(list(x), dropna=False, sort=False)["rotulo"]
        mascara &= grupos.transform("nunique").to_numpy() == 1
    if regra in {"sem_duplicatas", "sem_ambos"}:
        mascara &= ~chaves.duplicated().to_numpy()
    return indices[mascara]


def separar_divisoes(representacoes, y, grupos):
    divisor = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEMENTE)
    particoes = list(divisor.split(representacoes["C"], y, grupos))
    reserva = particoes[-1][1]
    divisoes = []
    for fold, (treino, validacao) in enumerate(particoes, 1):
        if fold != 5:
            treino = np.setdiff1d(treino, reserva)
            assert set(grupos.iloc[treino]).isdisjoint(grupos.iloc[reserva])
            assert set(grupos.iloc[validacao]).isdisjoint(grupos.iloc[reserva])
        assert set(grupos.iloc[treino]).isdisjoint(grupos.iloc[validacao])
        divisoes.append((fold, treino, validacao))
    return divisoes


In [ ]:
%%writefile "{codigo_tecnicas}"
"""Transformacoes ajustadas apenas no treino de cada divisao."""

import numpy as np
import pandas as pd
from sklearn import config_context
from sklearn.impute import KNNImputer, SimpleImputer
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder, OrdinalEncoder, RobustScaler, StandardScaler
from dados_etapa2 import NOMINAIS, SEMENTE, escolher_linhas


def transformar_numericas(treino, validacao, metodo, escala):
    treino = treino.to_numpy(dtype=np.float64, na_value=np.nan)
    validacao = validacao.to_numpy(dtype=np.float64, na_value=np.nan)
    # Uma entrada sem observacoes no ajuste recebe zero em ambas as parcelas.
    # Valores observados apenas na validacao nao participam do ajuste do KNN.
    if metodo == "knn":
        vazias = np.isnan(treino).all(axis=0)
        treino[:, vazias] = 0.0
        validacao[:, vazias] = 0.0
    if metodo == "mediana_indicador":
        tem_ausencia = np.isnan(treino).any(axis=0)
        indicadores_treino = np.isnan(treino[:, tem_ausencia])
        indicadores_val = np.isnan(validacao[:, tem_ausencia])
    if metodo == "knn":
        # Limita o custo de vizinhança a uma amostra sem rótulos da parte de treino.
        normalizador = StandardScaler().fit(treino)
        amostra = np.random.default_rng(SEMENTE).choice(len(treino), size=min(2000, len(treino)), replace=False)
        imputador = KNNImputer(n_neighbors=5, keep_empty_features=True)
        imputador.fit(normalizador.transform(treino[amostra]))
        with config_context(working_memory=48):
            treino = normalizador.inverse_transform(imputador.transform(normalizador.transform(treino)))
            validacao = normalizador.inverse_transform(imputador.transform(normalizador.transform(validacao)))
    else:
        estrategia = "mean" if metodo == "media" else "median"
        imputador = SimpleImputer(strategy=estrategia, keep_empty_features=True)
        treino, validacao = imputador.fit_transform(treino), imputador.transform(validacao)
    normalizadores = {"zscore": StandardScaler, "robusta": RobustScaler, "minmax": MinMaxScaler}
    if escala in normalizadores:
        normalizador = normalizadores[escala]()
        treino, validacao = normalizador.fit_transform(treino), normalizador.transform(validacao)
    if metodo == "mediana_indicador":
        treino = np.column_stack((treino, indicadores_treino))
        validacao = np.column_stack((validacao, indicadores_val))
    return treino, validacao


def transformar_categoricas(treino, validacao, metodo, codificacao):
    estrategia = "most_frequent" if metodo == "moda" else "constant"
    colunas = list(treino)
    colunas_treino, colunas_validacao = [], []
    for coluna in colunas:
        valores_treino = treino[[coluna]].to_numpy(dtype=object, na_value=np.nan)
        valores_validacao = validacao[[coluna]].to_numpy(dtype=object, na_value=np.nan)
        # A moda nao existe quando toda a coluna do ajuste esta ausente.
        # Nesse caso a categoria explicita mantem a posicao da coluna.
        estrategia_coluna = "constant" if pd.isna(valores_treino).all() else estrategia
        imputador = SimpleImputer(strategy=estrategia_coluna, fill_value="__ausente__")
        colunas_treino.append(imputador.fit_transform(valores_treino))
        colunas_validacao.append(imputador.transform(valores_validacao))
    treino = np.column_stack(colunas_treino)
    validacao = np.column_stack(colunas_validacao)
    if codificacao == "inteiros":
        codificador = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
        return codificador.fit_transform(treino), codificador.transform(validacao)
    cod_treino, cod_val = [], []
    for i, coluna in enumerate(colunas):
        agrupar = codificacao == "onehot_raros" and coluna == "service"
        codificador = OneHotEncoder(
            sparse_output=False,
            min_frequency=20 if agrupar else None,
            handle_unknown="infrequent_if_exist" if agrupar else "ignore",
        )
        cod_treino.append(codificador.fit_transform(treino[:, i:i + 1]))
        cod_val.append(codificador.transform(validacao[:, i:i + 1]))
    return np.column_stack(cod_treino), np.column_stack(cod_val)


def transformar(x, y, indices, validacao, config):
    indices = escolher_linhas(x, y, indices, config.linhas)
    treino, validacao = x.iloc[indices], x.iloc[validacao]
    categoricas = [c for c in NOMINAIS if c in x]
    numericas = x.columns.drop(categoricas).tolist()
    num_treino, num_val = transformar_numericas(treino[numericas], validacao[numericas], config.numericos, config.escala)
    cat_treino, cat_val = transformar_categoricas(treino[categoricas], validacao[categoricas], config.nominais, config.codificacao)
    return np.column_stack((num_treino, cat_treino)), np.column_stack((num_val, cat_val)), y[indices]


In [ ]:
%%writefile "{codigo_experimentos}"
"""Compara a mesma SVM RBF na CPU e no CUDA, com matrizes compartilhadas."""

import argparse
import json
from pathlib import Path
from time import perf_counter
from types import SimpleNamespace

import numpy as np


TOLERANCIAS = (1e-3, 1e-4)
PREPARACAO = SimpleNamespace(
    linhas="sem_duplicatas", numericos="mediana", nominais="ausencia",
    codificacao="onehot", escala="zscore",
)


def caminho_divisao(pasta, fold):
    return pasta / f"divisao_{fold}.npz"


def preparar(arquivo, pasta, folds):
    from dados_etapa2 import carregar_treino, preparar_base, separar_divisoes
    from tecnicas_etapa2 import transformar

    dados = carregar_treino(arquivo)
    y = dados["class"].to_numpy()
    representacoes, grupos = preparar_base(dados)
    divisoes = separar_divisoes(representacoes, y, grupos)
    for fold, treino, validacao in divisoes[:4]:
        if fold not in folds:
            continue
        x_tr, x_va, y_tr = transformar(representacoes["C"], y, treino, validacao, PREPARACAO)
        if not np.isfinite(x_tr).all() or not np.isfinite(x_va).all():
            raise ValueError(f"A divisão {fold} contém valores não finitos.")
        np.savez(
            caminho_divisao(pasta, fold), x_tr=x_tr, x_va=x_va,
            y_tr=(y_tr == "anomaly").astype(np.int32), y_va=(y[validacao] == "anomaly").astype(np.int32),
            gamma=1 / (x_tr.shape[1] * x_tr.var()),
        )
        print(f"Divisão {fold} preparada: {len(x_tr)} linhas de ajuste e {len(x_va)} de validação.", flush=True)


def medir_cpu(x_tr, x_va, y_tr, gamma, tol):
    from sklearn.svm import SVC

    modelo = SVC(kernel="rbf", C=10, gamma=gamma, tol=tol, max_iter=-1, cache_size=256)
    inicio = perf_counter()
    modelo.fit(x_tr, y_tr)
    ajuste = perf_counter() - inicio
    inicio = perf_counter()
    previsto = modelo.predict(x_va)
    previsao = perf_counter() - inicio
    return previsto, {
        "ajuste_s": ajuste, "previsao_s": previsao, "transferencia_s": 0.0,
        "iteracoes": int(modelo.n_iter_[0]),
        "estado_solver": "Convergiu" if modelo.fit_status_ == 0 else "Alerta de convergência",
    }


def medir_cuda(x_tr, x_va, y_tr, gamma, tol):
    import cupy as cp
    from cuml.svm import SVC

    modelo = SVC(
        kernel="rbf", C=10, gamma=gamma, tol=tol, max_iter=-1,
        cache_size=256, output_type="cupy",
    )
    sincronizar = cp.cuda.runtime.deviceSynchronize
    inicio = perf_counter()
    x_gpu = cp.asarray(np.asfortranarray(x_tr))
    y_gpu = cp.asarray(y_tr)
    sincronizar()
    transferencia = perf_counter() - inicio

    inicio = perf_counter()
    modelo.fit(x_gpu, y_gpu)
    sincronizar()
    ajuste = perf_counter() - inicio

    inicio = perf_counter()
    x_val_gpu = cp.asarray(np.asfortranarray(x_va))
    sincronizar()
    transferencia += perf_counter() - inicio

    inicio = perf_counter()
    previsto_gpu = modelo.predict(x_val_gpu)
    sincronizar()
    previsao = perf_counter() - inicio

    inicio = perf_counter()
    previsto = cp.asnumpy(previsto_gpu)
    transferencia += perf_counter() - inicio
    # O cuML pode parar pela tolerância ou por estagnação. Reportamos o código disponível.
    iteracoes = getattr(modelo, "n_iter_", None)
    estado = getattr(modelo, "fit_status_", None)
    if iteracoes is not None:
        iteracoes = int(cp.asnumpy(iteracoes).ravel()[0])
    return previsto, {
        "ajuste_s": ajuste, "previsao_s": previsao, "transferencia_s": transferencia,
        "iteracoes": iteracoes,
        "estado_solver": f"fit_status_={int(estado)}" if estado is not None else "Indisponível",
    }


def avaliar(pasta, folds, dispositivo):
    from sklearn.metrics import f1_score, precision_score, recall_score
    from threadpoolctl import threadpool_limits

    medir = medir_cpu if dispositivo == "cpu" else medir_cuda
    # Aquece ambos os backends e inicializa o contexto CUDA fora do cronômetro.
    x_curto = np.array([[0.0], [0.1], [0.9], [1.0]])
    y_curto = np.array([0, 0, 1, 1], dtype=np.int32)
    with threadpool_limits(limits=1):
        medir(x_curto, x_curto, y_curto, 1.0, TOLERANCIAS[0])
        linhas = []
        previsoes = {}
        for fold in folds:
            with np.load(caminho_divisao(pasta, fold), allow_pickle=False) as matriz:
                x_tr, x_va, y_tr, y_va = (matriz[k] for k in ("x_tr", "x_va", "y_tr", "y_va"))
                gamma = float(matriz["gamma"])
            for tol in TOLERANCIAS:
                previsto, tempo = medir(x_tr, x_va, y_tr, gamma, tol)
                previsto = np.asarray(previsto).reshape(-1)
                if previsto.shape != y_va.shape or not np.isin(previsto, [0, 1]).all():
                    raise ValueError(f"Previsões inválidas na divisão {fold}, tol={tol:g}.")
                resultado = {
                    "dispositivo": dispositivo.upper(), "fold": fold, "tol": tol, "gamma": gamma,
                    "linhas_ajuste": len(y_tr), "linhas_validacao": len(y_va),
                    "f1_macro": f1_score(y_va, previsto, average="macro"),
                    "recall_anomaly": recall_score(y_va, previsto, zero_division=0),
                    "precisao_anomaly": precision_score(y_va, previsto, zero_division=0),
                    "total_s": tempo["ajuste_s"] + tempo["previsao_s"] + tempo["transferencia_s"],
                    **tempo,
                }
                linhas.append(resultado)
                previsoes[f"d{fold}_{tol:g}"] = previsto
                print(
                    f"{dispositivo.upper()} divisão {fold}, tol={tol:g} · "
                    f"F1-macro {resultado['f1_macro']:.4f} · {resultado['total_s']:.1f} s",
                    flush=True,
                )
    np.savez(pasta / f"previsoes_{dispositivo}.npz", **previsoes)
    (pasta / f"resultados_{dispositivo}.json").write_text(json.dumps(linhas, ensure_ascii=False), encoding="utf-8")


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("modo", choices=["preparar", "cpu", "cuda", "concordancia"])
    parser.add_argument("pasta", type=Path)
    parser.add_argument("--folds", type=int, choices=range(1, 5), nargs="+", default=[1])
    parser.add_argument("--treino", type=Path)
    args = parser.parse_args()
    if args.modo == "preparar":
        preparar(args.treino, args.pasta, args.folds)
    elif args.modo == "concordancia":
        with np.load(args.pasta / "previsoes_cpu.npz") as cpu, np.load(args.pasta / "previsoes_cuda.npz") as cuda:
            concordancia = {k: float(np.mean(cpu[k] == cuda[k])) for k in cpu.files if k in cuda}
        print(json.dumps(concordancia))
    else:
        avaliar(args.pasta, args.folds, args.modo)


if __name__ == "__main__":
    main()

In [ ]:
import re
import shutil

def identificar_cuda():
    nvidia_smi = shutil.which("nvidia-smi")
    if not sys.platform.startswith("linux") or not nvidia_smi:
        return None
    consulta = subprocess.run(
        [nvidia_smi, "--query-gpu=name,compute_cap", "--format=csv,noheader"],
        capture_output=True, text=True,
    )
    if consulta.returncode or not consulta.stdout.strip():
        return None
    nome, capacidade = consulta.stdout.splitlines()[0].rsplit(",", 1)
    if float(capacidade) < 7.5:
        return None
    nvcc = shutil.which("nvcc")
    if nvcc is None and Path("/usr/local/cuda/bin/nvcc").is_file():
        nvcc = "/usr/local/cuda/bin/nvcc"
    versao = None
    if nvcc:
        toolkit = subprocess.run([nvcc, "--version"], capture_output=True, text=True)
        versao = re.search(r"release\s+(\d+)\.", toolkit.stdout)
    if versao is None:
        driver = subprocess.run([nvidia_smi], capture_output=True, text=True)
        versao = re.search(r"CUDA Version:\s*(\d+)\.", driver.stdout)
    if versao is None or int(versao.group(1)) < 12:
        return None
    return nome.strip(), min(int(versao.group(1)), 13)

gpu = identificar_cuda()
print(f"GPU detectada: {gpu[0]} · CUDA {gpu[1]}" if gpu else "Sessão disponível para a verificação em CPU.")

In [ ]:
# Para parear os tempos em todas as divisões, use [1, 2, 3, 4] também na CPU.
DIVISOES_CPU = [1]
DIVISOES_CUDA = [1, 2, 3, 4] if gpu else []
comando = [sys.executable, "-u", str(codigo_experimentos)]
folds_preparo = sorted(set(DIVISOES_CPU + DIVISOES_CUDA))
subprocess.run(
    [*comando, "preparar", str(trabalho), "--treino", str(treino_csv),
     "--folds", *map(str, folds_preparo)],
    check=True, env=ambiente,
)
subprocess.run(
    [*comando, "cpu", str(trabalho), "--folds", *map(str, DIVISOES_CPU)],
    check=True, env=ambiente,
)
resultados_cpu = json.loads((trabalho / "resultados_cpu.json").read_text(encoding="utf-8"))

## Execução em CUDA

O cuML é instalado, se necessário, em um diretório separado para o subprocesso da GPU. O notebook usa o pacote que corresponde à família CUDA da sessão. Preparo dos dados, instalação e inicialização da GPU ficam fora da medição. A comparação registra separadamente o ajuste, a previsão e a transferência dos dados entre CPU e GPU.

[Guia oficial da NVIDIA para Colab](https://docs.nvidia.com/datascience/deployment/26.10/platforms/colab/)

In [ ]:
resultados_cuda = []
concordancia = {}
falha_cuda = None
if gpu and DIVISOES_CUDA:
    nome_gpu, versao_cuda = gpu
    pacote = f"cuml-cu{versao_cuda}"
    versao_cuml = "26.6.0"
    bibliotecas_cuda = Path(tempfile.gettempdir()) / f"ia-cuda-{sys.implementation.cache_tag}-{pacote}-{versao_cuml}"
    ambiente_cuda = os.environ.copy()
    ambiente_cuda.update(PYTHONIOENCODING="utf-8", PYTHONNOUSERSITE="1", OPENBLAS_NUM_THREADS="1", OMP_NUM_THREADS="1")
    ambiente_cuda["PYTHONPATH"] = os.pathsep.join(
        filter(None, (str(bibliotecas_cuda), ambiente_cuda.get("PYTHONPATH")))
    )
    verificar_cuda = (
        "import importlib.metadata as m; import cupy as cp; from cuml.svm import SVC; "
        f"assert m.version({pacote!r}) == {versao_cuml!r}; "
        "assert cp.cuda.runtime.getDeviceCount() > 0"
    )
    try:
        verificar = [sys.executable, "-c", verificar_cuda]
        consulta = subprocess.run(verificar, env=ambiente_cuda, capture_output=True, text=True)
        if consulta.returncode:
            print(f"Preparando {pacote}=={versao_cuml}.", flush=True)
            subprocess.run(
                [sys.executable, "-m", "pip", "install", "--disable-pip-version-check",
                 "--upgrade", "--target", str(bibliotecas_cuda),
                 "--extra-index-url", "https://pypi.nvidia.com", f"{pacote}=={versao_cuml}"],
                env={**os.environ, "PYTHONNOUSERSITE": "1"}, check=True,
            )
            subprocess.run(verificar, env=ambiente_cuda, check=True)
        print(f"Executando com {nome_gpu} e {pacote}=={versao_cuml}.", flush=True)
        subprocess.run(
            [*comando, "cuda", str(trabalho), "--folds", *map(str, DIVISOES_CUDA)],
            check=True, env=ambiente_cuda,
        )
        resultados_cuda = json.loads((trabalho / "resultados_cuda.json").read_text(encoding="utf-8"))
        resposta = subprocess.check_output(
            [*comando, "concordancia", str(trabalho)], env=ambiente, text=True,
        )
        concordancia = json.loads(resposta)
    except subprocess.CalledProcessError as erro:
        falha_cuda = f"A execução CUDA terminou com código {erro.returncode}. Consulte a saída desta célula."
        print(falha_cuda)

## Métricas e tempos

Compare as duas tolerâncias dentro de cada backend. A tabela de desempenho entre CPU e CUDA usa apenas pares com a mesma divisão e a mesma tolerância. O total considera ajuste, previsão e transferências, com o contexto previamente inicializado. Uma execução por configuração dá uma medida inicial dependente da máquina e da carga da sessão.

In [ ]:
def tabela(colunas, linhas):
    linhas = [colunas, ["---"] * len(colunas), *linhas]
    display(Markdown("\n".join("| " + " | ".join(map(str, linha)) + " |" for linha in linhas)))

resultados = resultados_cpu + resultados_cuda
tabela(
    ("Backend", "Divisão", "Tolerância", "F1-macro", "Recall anomaly", "Precisão anomaly", "Iterações",
     "Ajuste (s)", "Previsão (s)", "Transferência (s)", "Total (s)", "Solver"),
    [(r["dispositivo"], r["fold"], f"{r['tol']:g}", f"{r['f1_macro']:.4f}", f"{r['recall_anomaly']:.2%}",
      f"{r['precisao_anomaly']:.2%}", r["iteracoes"] if r["iteracoes"] is not None else "—",
      f"{r['ajuste_s']:.3f}", f"{r['previsao_s']:.3f}", f"{r['transferencia_s']:.3f}",
      f"{r['total_s']:.3f}", r["estado_solver"]) for r in resultados],
)
for backend in ("CPU", "CUDA"):
    linhas_backend = [r for r in resultados if r["dispositivo"] == backend]
    for tol in sorted({r["tol"] for r in linhas_backend}, reverse=True):
        valores = [r["f1_macro"] for r in linhas_backend if r["tol"] == tol]
        print(f"{backend}, tolerância {tol:g}: F1-macro médio {sum(valores)/len(valores):.4f} em {len(valores)} divisão(ões).")

por_divisao_cuda = {(r["fold"], r["tol"]): r for r in resultados_cuda}
pares = []
for cpu in resultados_cpu:
    cuda = por_divisao_cuda.get((cpu["fold"], cpu["tol"]))
    if cuda:
        chave = f"d{cpu['fold']}_{cpu['tol']:g}"
        taxa = f"{concordancia[chave]:.2%}" if chave in concordancia else "—"
        pares.append((cpu["fold"], f"{cpu['tol']:g}", f"{cpu['total_s']/cuda['total_s']:.2f}×",
                      f"{cuda['f1_macro']-cpu['f1_macro']:+.4f}", taxa))
if pares:
    tabela(("Divisão", "Tolerância", "Tempo CPU / CUDA", "F1 CUDA − F1 CPU", "Previsões iguais"), pares)
else:
    print(falha_cuda or "Para medir a aceleração, execute o notebook em uma sessão Colab com GPU NVIDIA.")